
# Create Table - Table & Column Properties

We will see adding Table and Column Properties to the CREATE TABLE statement


## 1. Table Properties

1.1. COMMENT - allows you to document the purpose of the table

1.2. TBLPROPERTIES - used to specify table level metadata or configuration settings

In [0]:
%sql
DROP TABLE IF EXISTS demo.delta_lake.companies;

CREATE OR REPLACE TABLE demo.delta_lake.companies (
  company_name STRING, 
  founded_date DATE, 
  country STRING
)
COMMENT 'This table contains information about some of the successful tech companies' ;


DESCRIBE EXTENDED demo.delta_lake.companies;

-- COMMENT ON TABLE, shows as a description in the UI Unity Catalog // Just as a note if you want to do any operation on the Unity Catalog you have to connect to the server


### 1. Here, let's set delta.appendOnly = false and see if we can delete any records from the table 


Below, mentioned are some of the common tblproperties that we can set on the DeltaTable

    delta.appendOnly: Set to true to disable UPDATE and DELETE operations.
    delta.dataSkippingNumIndexedCols: Set to the number of leading column for which to collect and consider statistics.
    delta.deletedFileRetentionDuration: Set to an interval such as 'interval 7 days' to control when VACUUM is allowed to delete files.
    delta.logRetentionDuration: Set to an interval such as 'interval 60 days' to control how long history is kept for time travel queries.

In [0]:
%sql

INSERT INTO TABLE demo.delta_lake.companies 
VALUES ("Microsoft", "1975-04-04", "USA"), 
        ("Google", "1998-09-04", "USA"), 
        ("Amazon", "1994-07-05", "USA");


SELECT * FROM demo.delta_lake.companies;

In [0]:
%sql

DELETE FROM demo.delta_lake.companies 
WHERE company_name = "Amazon";


-- see, we can delete Amazon record from the table 
SELECT * FROM demo.delta_lake.companies;

In [0]:
%sql

-- Alright, Now we set the table properties as appendOnly = 'false'
ALTER TABLE demo.delta_lake.companies SET TBLPROPERTIES('delta.appendOnly' = 'true');


DESCRIBE EXTENDED demo.delta_lake.companies;

-- Now we can see the table properties has been changed

In [0]:
%sql

DELETE FROM demo.delta_lake.companies 
WHERE company_name = "Google";

-- See, now we cannot delete the Google records from the table 

In [0]:
%sql
ALTER TABLE demo.delta_lake.companies SET TBLPROPERTIES('delta.appendOnly' = 'false');


DESCRIBE EXTENDED demo.delta_lake.companies; 

 
## 2. Column Properties 

2.1 NOT NULL CONSTRAINTS - enforces data integrity and quality by ensuring that a specific column cannot contain NULL values 

2.2 COMMENT - documents the purpose or context of individual columns in a table  

In [0]:
%sql
DROP TABLE IF EXISTS demo.delta_lake.companies;

CREATE OR REPLACE TABLE demo.delta_lake.companies (
  company_name STRING NOT NULL, -- NOT NULL IS A CONSTRAINT
  founded_date DATE COMMENT 'The date when the company founded', -- At column level we can write the comments
  country STRING
)
COMMENT 'This table contains information about some of the successful tech companies'
TBLPROPERTIES('delta.appendOnly'='true');


DESCRIBE EXTENDED demo.delta_lake.companies;


### 2.3 Generated Columns - Derived or computed columns, whose value are computed at the time of inserting a new records

    2.3.1 Generated Identity Columns - used to generate an identity for example a surrogate key
    2.3.2 Generated Computed Columns - automatically calculate and store derived values based on the other columns in the same



#### 2.3.1 Generated Identity Columns

GENERATED {ALWAYS | BY DEFAULT} AS IDENTITY [ ( [ START WITH start ] [INCREMENT BY step] ) ]

In [0]:
%sql
DROP TABLE IF EXISTS demo.delta_lake.companies;

CREATE OR REPLACE TABLE demo.delta_lake.companies(
  company_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  company_name STRING NOT NULL, -- NOT NULL IS A CONSTRAINT
  founded_date DATE COMMENT 'The date when the company founded', -- At column level we can write the comments
  country STRING
)
COMMENT 'This table contains information about some of the successful tech companies'
TBLPROPERTIES('delta.appendOnly'='true');

In [0]:
%sql

-- You have to specify the column name in this case for sure, otherwise it throws the error
INSERT INTO demo.delta_lake.companies (company_name, founded_date, country)
VALUES ("Apple", "1976-04-01", "USA"),
      ("Microsoft", "1975-04-04", "USA"), 
      ("Google", "1998-09-04", "USA"), 
      ("Amazon", "1994-07-05", "USA");

In [0]:
%sql
SELECT * FROM demo.delta_lake.companies;

-- We can it automatically increment the id column and generated as well


### 2.3.2 Generated Computed Columns

GENERATED ALWAYS AS ( expr )

expr may be composed of literals, column identifiers within the table, and deterministic, built-in SQL functions or operators except:

1. Aggregate functions
2. Analytic window functions
3. Ranking window functions
4. Table valued generator functions

Also expr must not contain any subquery

In [0]:
%sql
DROP TABLE IF EXISTS demo.delta_lake.companies;

CREATE OR REPLACE TABLE demo.delta_lake.companies(
  company_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  company_name STRING NOT NULL, -- NOT NULL IS A CONSTRAINT
  founded_date DATE COMMENT 'The date when the company founded', -- At column level we can write the comments
  founded_year INT GENERATED ALWAYS AS (YEAR(founded_date)) COMMENT 'The year when the company founded a computed field', 
  country STRING
)
COMMENT 'This table contains information about some of the successful tech companies'
TBLPROPERTIES('delta.appendOnly'='true');

In [0]:
%sql
INSERT INTO demo.delta_lake.companies (company_name, founded_date, country)
VALUES ("Apple", "1976-04-01", "USA"),
      ("Microsoft", "1975-04-04", "USA"), 
      ("Google", "1998-09-04", "USA"), 
      ("Amazon", "1994-07-05", "USA");

In [0]:
%sql

-- Perfect, we can also create the computed column automatically while inserting the columns in the table itself
SELECT * FROM demo.delta_lake.companies;